# Pipeline de Séries Temporais: SARIMA Vs. Base Models

## Etapa 1: Imports

In [ ]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.filterwarnings('ignore')
warnings.simplefilter('ignore', ConvergenceWarning)


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX

## Etapa 2: Arquivos

In [ ]:
ARQUIVOS = [
    {
        'nome':    'spacy',
        'path':    'serie_atv_6.csv',
        'coluna':  'spacy',
        'index': 'month',
        'start_date': '2015-01-01',
    }
]


## Etapa 3: Funções

In [ ]:
def carregar_serie(arquivo):
    df = pd.read_csv(arquivo['path'], usecols=[arquivo['index'], arquivo['coluna']])
    df['month'] = pd.to_datetime(df['month'],format='%y-%b')
    df = df.sort_values('month')
    df = df.set_index('month')
    df = df.asfreq('MS')
    df = df.loc[arquivo['start_date']:].dropna()
    serie = df[arquivo['coluna']].dropna()
    print(df.isnull().sum())
    print(df.head())
    print('\nInformações da base:')
    print(df.info())
    print('\nEstatísticas:')    
    print(df.describe())
    print(f'Quantidade de observações: {len(df)}')
    print(f'Período da série: {df.index.min()} até {df.index.max()}')
    return serie

In [ ]:
def plotar_serie(serie, nome):
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(serie.values, color='#2563eb', linewidth=1.2)
    ax.set_title(f'{nome}', fontweight='bold')
    ax.set_xlabel('Tempo')
    ax.set_ylabel('Valor')
    plt.tight_layout()
    plt.show()

In [ ]:
def plotar_acf_pacf(serie, nome, n_lags=30):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    plot_acf(serie,  lags=n_lags, ax=axes[0], title=f'ACF: {nome}')
    plot_pacf(serie, lags=n_lags, ax=axes[1], title=f'PACF: {nome}')
    for ax in axes:
        ax.set_xlabel('Lag')
    plt.tight_layout()
    plt.show()

In [ ]:
def teste_adf(serie, alpha=0.05):
    resultado_adf = adfuller(serie)
    estatistica_adf = resultado_adf[0]
    p_valor = resultado_adf[1]
    estacionaria = p_valor <= alpha
    print(f'ADF: resultado={estatistica_adf:.4f}  p={p_valor:.4f}'
          f' = {"Estacionária ✅" if estacionaria else "Não Estacionária ❌"}')
    return estacionaria

In [ ]:
def teste_kpss(serie, alpha=0.05):
      resultado_kpss = kpss(serie)
      estatistica_kpss = resultado_kpss[0]
      p_valor = resultado_kpss[1]
      estacionaria = p_valor > alpha
      print(f'KPSS: resultado={estatistica_kpss:.4f}  p={p_valor:.4f}'
            f' = {"Estacionária ✅" if estacionaria else "Não Estacionária ❌"}')
      return estacionaria

In [ ]:
def testar_sazonalidade(serie):
    melhor_m = []
    for m in range(2, 13):
        stl = STL(serie, period=m, robust=True).fit()
        forca_sazonalidade = 1 - (np.var(stl.resid) / np.var(stl.seasonal + stl.resid))
        melhor_m.append((m, forca_sazonalidade))
    m_definido, forca_sazonalidade = max(melhor_m, key=lambda x: x[1])

    fig = stl.plot()
    fig.set_size_inches(20, 10)
    plt.tight_layout()
    plt.show()

    print(f'Força da Sazonalidade para {m_definido}: {forca_sazonalidade:.5f}')
    return m_definido

In [ ]:
def avaliar_estacionariedade(serie):
    adf = teste_adf(serie)
    kpss = teste_kpss(serie)
    if adf and kpss:
        print('Conclusão: A série é estacionária ✅')
        return True
    elif not adf and not kpss:
        print('Conclusão: A série é não estacionária ❌')
    elif adf and not kpss:
        print('Conclusão: Conflito ADF estacionário / KPSS não estacionário - tratada como não estacionária ⚠️')
    else:
        print('Conclusão: Conflito ADF não estacionário / KPSS estacionário - tratada como não estacionária ⚠️')
    return False

In [ ]:
def tornar_estacionaria(serie, max_diff=3):
    serie_diferenciacao = serie.copy()
    resultado = False
    d = 0
    while resultado == False and d <= max_diff:
        print(f'\nTestando série com d={d}')
        resultado = avaliar_estacionariedade(serie_diferenciacao)
        if resultado:
            print(f'\nSérie estacionária com d={d}')
            return serie_diferenciacao, d
        print('\nSérie não estacionária.')
        print('Aplicando diferenciação...\n')
        serie_diferenciacao = (serie_diferenciacao.diff().dropna())        
        d += 1


In [ ]:
def formatar(texto, nivel=2):
    prefixos = {1: '\n' + '═'*65, 2: '─'*55, 3: '·'*45}
    sep = prefixos.get(nivel, '')
    if nivel == 1:
        print(f'{sep}\n  {texto}\n{sep}')
    else:
        print(f'\n{sep}\n  {texto}\n{sep}')

In [ ]:
def dividir_serie(serie, proporcao_teste=0.2):
    n_teste = max(1, int(len(serie) * proporcao_teste))
    treino  = serie.iloc[:-n_teste].reset_index(drop=True)
    teste   = serie.iloc[-n_teste:].reset_index(drop=True)
    print(f'Total: {len(serie)} obs  |  Treino: {len(treino)}  |  Teste: {len(teste)}')
    return treino, teste

In [ ]:
def _rolling_prev(treino, teste, fn):
    serie = pd.concat([treino, teste], ignore_index=True)
    return fn(serie).shift(1).iloc[len(treino):].values

In [ ]:
def modelo_media_historica(treino, teste):
    return np.full(len(teste), treino.mean())

In [ ]:
def modelo_media_acumulada(treino, teste):
    return _rolling_prev(treino, teste, lambda s: s.expanding().mean())

In [ ]:
def modelo_sma(treino, teste, janela):
    return _rolling_prev(treino, teste, lambda s: s.rolling(window=int(janela)).mean())

In [ ]:
def modelo_ema(treino, teste, alpha):
    return _rolling_prev(treino, teste, lambda s: s.ewm(alpha=float(alpha), adjust=False).mean())

In [ ]:
def modelo_seasonal_naive(treino, teste, s):
    return _rolling_prev(treino, teste, lambda s_: s_.shift(int(s)))

In [ ]:
def modelo_naive_drift(treino, teste, k):
    serie = pd.concat([treino, teste], ignore_index=True)
    drift = (serie.shift(1) - serie.shift(1 + int(k))) / int(k)
    return (serie.shift(1) + drift).iloc[len(treino):].values

In [ ]:
def _otimizar_param(treino, modelo_fn, params):
    n_val    = max(2, int(len(treino) * 0.2))
    tr_int   = treino.iloc[:-n_val].reset_index(drop=True)
    val_int  = treino.iloc[-n_val:].reset_index(drop=True)

    melhor_param, melhor_mae = params[0], np.inf
    for p in params:
        try:
            prev     = modelo_fn(tr_int, val_int, p)
            mascara  = ~np.isnan(prev)
            if mascara.sum() == 0:
                continue
            mae = np.mean(np.abs(prev[mascara] - val_int.values[mascara]))
            if mae < melhor_mae:
                melhor_mae   = mae
                melhor_param = p
        except Exception:
            continue
    return melhor_param

In [ ]:
def aplicar_base_models(treino, teste, sazonalidade):
    max_k   = max(2, min(len(treino) // 4, 15))
    janelas = list(range(1, max_k + 1))
    alphas  = np.round(np.arange(0.1, 1.0, 0.1), 2).tolist()

    k_sma   = _otimizar_param(treino, modelo_sma,         janelas)
    alpha   = _otimizar_param(treino, modelo_ema,         alphas)
    k_drift = _otimizar_param(treino, modelo_naive_drift, janelas)

    print(f'  Hiperparâmetros selecionados (MAE no treino):')
    print(f'    SMA            → janela = {k_sma}')
    print(f'    EMA            → alpha  = {alpha}')
    print(f'    Naive c/ Drift → k      = {k_drift}')
    print(f'    Seasonal Naive → s      = {sazonalidade}')

    previsoes = {
        'Média Histórica':                    modelo_media_historica(treino, teste),
        'Média Acumulada':                    modelo_media_acumulada(treino, teste),
        f'SMA (k={k_sma})':                   modelo_sma(treino, teste, k_sma),
        f'EMA (α={alpha})':                   modelo_ema(treino, teste, alpha),
        f'Seasonal Naive (s={sazonalidade})': modelo_seasonal_naive(treino, teste, sazonalidade),
        f'Naive c/ Drift (k={k_drift})':      modelo_naive_drift(treino, teste, k_drift),
    }
    return previsoes

In [ ]:
def plotar_base_models(treino, teste, previsoes, nome):
    serie_completa = pd.concat([treino, teste], ignore_index=True)
    n_treino = len(treino)
    x_teste  = np.arange(n_treino, n_treino + len(teste))

    cores   = ['#2563eb', '#16a34a', '#d97706', '#dc2626',
               '#7c3aed', '#0891b2']
    estilos = ['--', '-.', ':', '--', '-.', ':']

    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(serie_completa.values, color='#1e293b', linewidth=1.4,
            label='Série Original')
    ax.axvline(x=n_treino - 1, color='#94a3b8', linestyle='--',
               linewidth=1, label='Início do Teste')

    for (nome_modelo, prev), cor, estilo in zip(previsoes.items(), cores, estilos):
        mascara = ~np.isnan(prev)
        x_plot  = x_teste[mascara]
        y_plot  = prev[mascara]
        ax.plot(x_plot, y_plot, linestyle=estilo, color=cor,
                linewidth=1.6, label=nome_modelo)

    ax.set_title(f'Base Models — {nome}', fontweight='bold')
    ax.set_xlabel('Tempo')
    ax.set_ylabel('Valor')
    ax.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()

In [ ]:
def treinar_sarima(ytrain, m):
    resultados = []
    p_range = range(0, 3)
    d_range = range(0, 2)
    q_range = range(0, 3)
    P_range = range(0, 3)
    D_range = range(0, 2)
    Q_range = range(0, 3)
    for p in p_range:
        for d in d_range:
            for q in q_range:
                for P in P_range:
                    for D in D_range:
                        for Q in Q_range:
                            order = (p, d, q)
                            seasonal_order = (P, D, Q, m)
                            try:
                                model = SARIMAX(ytrain, order=order, seasonal_order=seasonal_order, enforce_stationarity=False, enforce_invertibility=False )
                                res = model.fit( disp=False, maxiter=100)
                                resultados.append({
                                    "model": res,
                                    "order": order,
                                    "seasonal_order": seasonal_order,
                                    "aic": res.aic,
                                    "bic": res.bic
                                })
                            except Exception:
                                continue
    resultados = sorted(resultados, key=lambda x: x["aic"])
    return resultados

In [ ]:
def tentar_sarima(resultados):
    melhor_bic = float('inf')
    melhor_modelo = None
    for resultado in resultados:
        try:
            order = resultado["order"]
            seasonal_order = resultado["seasonal_order"]
            model = resultado["model"]
            if model.bic < melhor_bic:
                melhor_bic = model.bic
                melhor_modelo = (order, seasonal_order, model)
        except Exception:
            continue
    print(f"Melhor modelo encontrado: SARIMA{melhor_modelo[0]}x{melhor_modelo[1]} - BIC: {melhor_bic:.2f}")
    return melhor_modelo

In [ ]:
def roling_backtest(y, melhor_modelo, h=1, initial=0.7):
    order, seasonal_order = melhor_modelo[0], melhor_modelo[1]
    n0 = int(len(y) * initial)
    erros = []
    
    for i in range(n0, len(y) - h + 1):
        train = y.iloc[:i]
        test = y.iloc[i : i + h]
        try:
            model = SARIMAX(train, order=order, seasonal_order=seasonal_order,
                            enforce_stationarity=True, enforce_invertibility=True)
            res = model.fit(disp=False)
            pred = res.forecast(steps=h)            
            erro = np.abs(test.values - pred.values).mean()
            if not np.isnan(erro) and not np.isinf(erro):
                erros.append(erro)            
        except Exception:
            continue
    mae_medio = np.mean(erros) if len(erros) > 0 else np.nan
    return mae_medio

In [ ]:
def rolling_forecast(treino, teste, modelo_sarima):
    historico = list(treino.values)
    previsoes = []
    
    if isinstance(modelo_sarima, dict):
        order = modelo_sarima.get('order', (1, 1, 1))
        seasonal_order = modelo_sarima.get('seasonal_order', (0, 0, 0, 0))
    else:
        try:
            order = modelo_sarima.model.order
            seasonal_order = modelo_sarima.model.seasonal_order
        except AttributeError:
            order = (1, 1, 1)
            seasonal_order = (0, 0, 0, 0)

    for i in range(len(teste)):
        try:
            modelo = SARIMAX(historico, order=order, seasonal_order=seasonal_order)
            modelo_fit = modelo.fit(disp=False)
            yhat = modelo_fit.forecast()[0]
        except Exception:
            yhat = np.nan
        previsoes.append(yhat)
        historico.append(teste.iloc[i])
        
    return np.array(previsoes)

In [ ]:
def comparar_modelos(treino, teste, previsoes_base, previsoes_sarima):
    previsoes_todas = previsoes_base.copy()
    previsoes_todas['SARIMA'] = previsoes_sarima
    plotar_base_models(treino, teste, previsoes_todas, f'Comparação Geral')



In [ ]:
def avaliar_residuos(modelo, nome):
    residuos = modelo.resid
    plt.figure(figsize=(10, 4))
    plt.plot(residuos, color='#e11d48', label='Resíduos')
    plt.title(f'Resíduos do Modelo AR: {nome}', fontweight='bold')
    plt.xlabel('Tempo')
    plt.ylabel('Valor')
    plt.legend()
    plt.tight_layout()
    plt.show()

    max_lag = min(10, len(residuos) // 2)
    lb_test = acorr_ljungbox(residuos, lags=range(1, max_lag + 1), return_df=True)
    print(lb_test.tail())
    print(f'Ljung-Box Test: p-value={lb_test["lb_pvalue"].values[-1]:.4f}')

In [ ]:
def realizar_teste_sarima(modelo, y_test):
    try:
        pred = modelo.get_forecast(steps=len(y_test)).predicted_mean.values
        erro = np.abs(y_test.values - pred).mean()
        print(f'MAE teste: {erro:.4f}')
    except Exception as e:
        print(f'Erro ao realizar teste SARIMA: {e}')


In [ ]:
def plotar_resultados(serie, ytest, pred, nome):
    plt.figure(figsize=(10, 4))
    plt.plot(
        serie.index,
        serie.values,
        label='Real',
        color='#2563eb'
    )
    plt.plot(
        ytest.index,
        pred,
        label='Previsto',
        color='#e11d48'
    )
    plt.title(f'Previsão vs Real: {nome}', fontweight='bold')
    plt.xlabel('Tempo')
    plt.ylabel('Valor')
    plt.legend()
    plt.grid(True)

    plt.show()

In [ ]:
def plotar_acf_residuos(modelo, nome, n_lags=30):
    residuos = modelo.resid
    max_lag = min(10, n_lags)
    lb_test = acorr_ljungbox(residuos, lags=range(1, max_lag + 1), return_df=True)
    print(lb_test.tail())
    plt.tight_layout()
    plt.show()

In [ ]:
def executar_sarima(serie, nome,m):
    formatar(f'Pipeline SARIMA - {nome}', nivel=1)
    plotar_serie(serie, nome)
    plotar_acf_pacf(serie, nome)
    tornar_estacionaria(serie)
    treino, teste = dividir_serie(serie)
    resultados_sarima = treinar_sarima(treino, m)
    melhor_modelo = tentar_sarima(resultados_sarima)
    mae_backtest = roling_backtest(serie, melhor_modelo)
    print(f'MAE médio do backtest: {mae_backtest:.4f}')
    modelo_sarima = melhor_modelo[2]
    previsoes_sarima = rolling_forecast(treino, teste, modelo_sarima)
    return previsoes_sarima

In [ ]:
def calcular_mae():
    raise NotImplementedError("Implementar calcular_mae()")

## Etapa 4: Pipeline

In [ ]:
def pipeline_sarima(arquivo, proporcao_teste=0.2):
    nome         = arquivo['nome']
    

    formatar(f'PIPELINE SARIMA: {nome}', 1)

    formatar('Carregando e visualizando a série...')
    serie = carregar_serie(arquivo)
    sazonalidade = testar_sazonalidade(serie)
    
    plotar_serie(serie, nome)
    plotar_acf_pacf(serie, nome)

    formatar('Dividindo em treino e teste...')
    treino, teste = dividir_serie(serie, proporcao_teste)

    formatar('Avaliando estacionariedade do treino...')
    estacionaria = avaliar_estacionariedade(treino)

    formatar('Treinando Base Models...')
    previsoes_base = aplicar_base_models(treino, teste, sazonalidade)
    plotar_base_models(treino, teste, previsoes_base, nome)

    formatar('Treinando modelo SARIMA...')
    previsoes_sarima = executar_sarima(serie, nome, sazonalidade)

        # formatar('Realizando Rolling Forecast...')
        # previsoes_sarima = rolling_forecast()

    formatar('Comparando SARIMA vs Base Models...')
    comparar_modelos(treino, teste, previsoes_base, previsoes_sarima)

    formatar('Calculando MAE e testes...')
    resultados = calcular_mae()

    return resultados

## Etapa 5: Execução

In [ ]:
resultados_gerais = {}

for arquivo in ARQUIVOS:
    resultados_gerais[arquivo['nome']] = pipeline_sarima(arquivo)